## ▶ Run Online — No Installation Needed

| Platform | Link |
|---|---|
| **Binder** (no account) | [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Piyushjhu/HELIX_Toolbox/main?labpath=examples%2F01_full_pipeline_cli.ipynb) |
| **Google Colab** | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Piyushjhu/HELIX_Toolbox/blob/main/examples/01_full_pipeline_cli.ipynb) |
| **GitHub Codespaces** | [![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/Piyushjhu/HELIX_Toolbox) |

**Using your own data?** Run the setup cell below — it will show an upload widget when running in Binder or Colab. On Codespaces, drag-and-drop your CSVs into the file explorer then update the path variables in the next cell.

In [ ]:

# ── Cloud / Online environment setup ──────────────────────────────────────
# This cell auto-detects Binder, Google Colab, GitHub Codespaces, and local
# environments.  It installs dependencies, sets headless Qt/matplotlib, and
# wires REPO_ROOT so all imports work without changes to subsequent cells.
# ──────────────────────────────────────────────────────────────────────────
import os, sys, subprocess

# ── Detect environment ─────────────────────────────────────────────────────
try:
    import google.colab
    _ENV = "colab"
except ImportError:
    _ENV = "binder" if os.environ.get("BINDER_SERVICE_HOST") else "local"

print(f"Detected environment: {_ENV}")

# ── Install / configure ────────────────────────────────────────────────────
if _ENV in ("colab", "binder"):
    # Headless Qt (no display needed) and non-interactive matplotlib
    os.environ["QT_QPA_PLATFORM"] = "offscreen"
    os.environ["MPLBACKEND"] = "Agg"

if _ENV == "colab":
    # Fixed absolute path — prevents double/triple nesting when cell is re-run
    REPO_ROOT = "/content/HELIX_Toolbox"
    if not os.path.isdir(REPO_ROOT):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Piyushjhu/HELIX_Toolbox.git",
             REPO_ROOT],
            check=True
        )
    else:
        # Repo already cloned — pull latest so sample data / fixes are current
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only"], check=False)
    # Install dependencies using the absolute path — no os.chdir() needed
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         os.path.join(REPO_ROOT, "requirements.txt")],
        check=False
    )
elif _ENV == "binder":
    # On Binder the repo is already at the working directory root
    REPO_ROOT = os.path.abspath("..")
else:
    # Local — adjust if your notebook is not inside the examples/ subfolder
    REPO_ROOT = os.path.abspath("..")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# ── Bundled sample data (used when no local files are provided) ────────────
SAMPLE_DIR = os.path.join(REPO_ROOT, "input_data", "C1_files")
_sample_files = sorted(f for f in os.listdir(SAMPLE_DIR) if f.endswith(".csv")) if os.path.isdir(SAMPLE_DIR) else []
print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"Sample data: {SAMPLE_DIR}")
print(f"  files    : {_sample_files}")

# ── Upload widget (Binder / Colab only) ────────────────────────────────────
_uploaded_dir = None
if _ENV == "colab":
    print("\nTo upload YOUR OWN data:  from google.colab import files; uploaded = files.upload()")
elif _ENV == "binder":
    print("\nTo use YOUR OWN data, use the Jupyter file-browser (left panel) to")
    print("upload CSVs, then set INPUT_DIR in the next cell to point to them.")
else:
    print("\nRunning locally — edit path variables in the next cell as needed.")


# Example 1 — Full Pipeline: Raw PDV → Spall Strength (CLI)

This notebook walks through a complete end-to-end HELIX Toolbox run:

```
Raw PDV oscilloscope CSV
        ↓  ALPSS
Smoothed velocity trace + uncertainty CSV
        ↓  SPADE
Spall strength / strain rate / shock stress / HEL summary CSV + plots
```

The run is driven entirely by a YAML config file — the same workflow you
would use in batch / HPC mode via the CLI:
```bash
python helix_cli_runner.py --config my_experiment.yml
```

---
**Before running:** update the path variables in the next cell.

In [ ]:
import os, sys

# ── USER PATHS — edit these for local runs ────────────────────────────────
# On Binder/Colab, REPO_ROOT and SAMPLE_DIR are already set by the setup cell
# above.  Override INPUT_DIR here to point to your own uploaded files.
try:
    REPO_ROOT   # set by setup cell
except NameError:
    REPO_ROOT = os.path.abspath("..")

try:
    SAMPLE_DIR  # set by setup cell
except NameError:
    SAMPLE_DIR = os.path.join(REPO_ROOT, "input_data", "C1_files")

# Default to bundled sample data; replace with your own folder path:
INPUT_DIR    = SAMPLE_DIR                    # ← your PDV CSV folder
PARAM_FOLDER = None                          # ← metadata xlsx/csv folder (or None)
OUTPUT_DIR   = os.path.join(REPO_ROOT, "examples", "figures", "example_01_output")
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("Repo root :", REPO_ROOT)
print("Input dir :", INPUT_DIR)
print("Output dir:", OUTPUT_DIR)

## 1. Build the config

We load `helix_master_config.json` from the repo as the base (guaranteeing
all required ALPSS/SPADE parameters are present), then override only the
run-specific fields (paths, material, analysis mode).

In [ ]:
import json as _json
from helix_analysis_toolbox import save_config_to_file

# ── Load the master config as base — guarantees ALL required params are present ──
_master_path = os.path.join(REPO_ROOT, "helix_master_config.json")
with open(_master_path) as _f:
    config = _json.load(_f)

# ── Override run-specific settings ───────────────────────────────────────────────
config["cli_settings"].update({
    "input_dir":     INPUT_DIR,
    "input_pattern": "*.csv",
    "output_dir":    OUTPUT_DIR,
    "param_folder":  PARAM_FOLDER,
    "analysis_mode": "both",    # both | alpss_only | spade_only
    "spade_mode":    "auto",
    "input_files":   None,
    "spade_input_files":    None,
    "spade_input_dir":      None,
    "spade_input_pattern":  "*--vel-smooth-with-uncert.csv",
})

# ── Material: update if your sample is not Cu ─────────────────────────────────
# CP-Ti values — change to match your experiment material
config["alpss_config"].update({
    "C0":     5020.0,   # bulk wave speed (m/s)
    "density": 4510.0,  # density (kg/m³)
    "display_plots": "no",
    "save_all_plots": "no",
})
config["spade_config"].update({
    "density":           4510.0,
    "acoustic_velocity": 5020.0,
    "spall_start_time_ns": 20.0,
    "spall_end_time_ns":   60.0,
    "show_plots": False,
    "skip_unknown_material_traces": False,
})

# ── Save config to YAML for the CLI ──────────────────────────────────────────
config_path = os.path.join(OUTPUT_DIR, "run_config.yml")
ok, msg = save_config_to_file(config, config_path)
print(msg)


## 2. Run the CLI

We call `helix_cli_runner.py` as a subprocess so the output streams live
to the notebook cell output exactly as it would in a terminal.

In [ ]:
import subprocess, os

_runner = os.path.join(REPO_ROOT, "helix_cli_runner.py")
print(f"REPO_ROOT   : {REPO_ROOT}")
print(f"runner path : {_runner}  (exists={os.path.isfile(_runner)})")
print(f"config path : {config_path}  (exists={os.path.isfile(config_path)})")
print("-" * 60)

result = subprocess.run(
    [sys.executable, _runner, "--config", config_path],
    capture_output=True,    # capture so we can print stderr on failure
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("── STDERR ──────────────────────────────────────────────────")
    print(result.stderr)
print(f"\nExit code: {result.returncode}")

## 3. Inspect the output summary

In [ ]:
import pandas as pd, os, glob as _glob

spade_dir = os.path.join(OUTPUT_DIR, "SPADE_analysis")

# SPADE writes spall results to enhanced_spall_summary.csv (or spall_summary.csv).
# velocity_shots_summary.csv only has velocity data, not spall/stress.
# Try files in priority order.
_candidates = [
    os.path.join(spade_dir, "enhanced_spall_summary.csv"),
    os.path.join(spade_dir, "spall_summary.csv"),
    os.path.join(spade_dir, "velocity_shots_summary.csv"),
]
summary_path = next((p for p in _candidates if os.path.exists(p)), None)

# Also list every CSV in the SPADE directory for diagnostics
print("Files in SPADE_analysis/:")
for f in sorted(_glob.glob(os.path.join(spade_dir, "*.csv"))):
    print(f"  {os.path.basename(f)}")

if summary_path:
    df = pd.read_csv(summary_path)
    print(f"\nLoaded {len(df)} row(s) from: {os.path.basename(summary_path)}")
    print("Columns:", list(df.columns))
    display(df)
else:
    df = None
    print("\nNo summary CSV found — check the run output above for errors.")


## 4. Quick result plots

In [ ]:
import matplotlib.pyplot as plt
import glob as _glob
import numpy as np
%matplotlib inline

if df is None or len(df) == 0:
    print("No data to plot — run Cell 9 first.")
else:
    # ── column discovery ──────────────────────────────────────────────────
    def _find(df, *candidates):
        for c in candidates:
            for col in df.columns:
                if c.lower() in col.lower():
                    return col
        return None

    col_stress  = _find(df, "shock_stress", "Shock_Stress", "Peak_Stress", "peak_stress")
    col_spall   = _find(df, "spall_strength", "Spall_Strength", "spall_str")
    col_strrate = _find(df, "strain_rate", "Strain_Rate")
    col_mat     = _find(df, "material", "Material", "Sample")
    col_vel     = _find(df, "peak_velocity", "Peak_Velocity", "max_velocity")

    print(f"shock_stress col  : {col_stress}")
    print(f"spall_strength col: {col_spall}")
    print(f"strain_rate col   : {col_strrate}")
    print(f"material col      : {col_mat}")
    print(f"peak_velocity col : {col_vel}")

    has_spall  = col_spall  and col_strrate and df[col_spall].notna().any()
    has_stress = col_stress and df[col_stress].notna().any()

    # ── load velocity traces (ALPSS output) ───────────────────────────────
    vel_files = sorted(_glob.glob(os.path.join(OUTPUT_DIR, "*--vel-smooth-with-uncert.csv")))
    print(f"\nVelocity files found: {len(vel_files)}")

    # ── figure layout ─────────────────────────────────────────────────────
    n_panels = 1 + int(has_spall) + int(has_stress and has_spall)  # vel + up to 2 spall panels
    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5))
    if n_panels == 1:
        axes = [axes]

    # Panel 0 — free surface velocity traces ──────────────────────────────
    ax = axes[0]
    colors = plt.cm.tab10.colors
    for i, vf in enumerate(vel_files):
        try:
            vdf = pd.read_csv(vf)
            # Column names produced by ALPSS
            t_col  = next((c for c in vdf.columns if "time" in c.lower()), vdf.columns[0])
            v_col  = next((c for c in vdf.columns if "velocity" in c.lower() and "uncert" not in c.lower()
                           and "smooth" in c.lower()), None)
            if v_col is None:
                v_col = next((c for c in vdf.columns if "velocity" in c.lower()
                              and "uncert" not in c.lower()), vdf.columns[1])
            u_col  = next((c for c in vdf.columns if "uncert" in c.lower()), None)

            t = vdf[t_col].values * 1e9          # s → ns
            v = vdf[v_col].values
            label = os.path.basename(vf).split("--vel")[0][-20:]   # last 20 chars of stem

            color = colors[i % len(colors)]
            ax.plot(t, v, lw=1.2, color=color, label=label)
            if u_col is not None:
                u = vdf[u_col].values
                ax.fill_between(t, v - u, v + u, alpha=0.2, color=color)
        except Exception as e:
            print(f"  Could not plot {os.path.basename(vf)}: {e}")

    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("Free surface velocity (m/s)")
    ax.set_title("Free surface velocity traces")
    if len(vel_files) > 1:
        ax.legend(fontsize=7, loc="best")
    ax.grid(True, alpha=0.3)

    # Panel 1 — spall strength vs strain rate ─────────────────────────────
    if has_spall:
        ax = axes[1]
        grps = df.groupby(col_mat) if col_mat else [(None, df)]
        for mat, grp in grps:
            ax.scatter(grp[col_strrate], grp[col_spall], label=mat, s=60)
        if col_mat:
            ax.legend()
        ax.set_xlabel("Strain rate (s⁻¹)")
        ax.set_ylabel("Spall strength (GPa)")
        ax.set_title("Spall strength vs strain rate")
        ax.set_xscale("log")
        ax.grid(True, alpha=0.3)

    # Panel 2 — spall strength vs shock stress ────────────────────────────
    if has_stress and has_spall:
        ax = axes[2]
        grps = df.groupby(col_mat) if col_mat else [(None, df)]
        for mat, grp in grps:
            ax.scatter(grp[col_stress], grp[col_spall], label=mat, s=60)
        if col_mat:
            ax.legend()
        ax.set_xlabel("Shock stress (GPa)")
        ax.set_ylabel("Spall strength (GPa)")
        ax.set_title("Spall strength vs shock stress")
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig_path = os.path.join(OUTPUT_DIR, "example_01_summary_plots.png")
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved to {fig_path}")


## 5. Show a generated individual trace plot

HELIX saves a per-trace spall detection plot for every file when
`plot_individual: true`.  Pick one to display here.

In [ ]:
import glob as _glob
from IPython.display import Image, display as ipy_display

# Find the first spall plot
spall_plots = _glob.glob(os.path.join(spade_dir, "spall_plots", "*.png"))
if spall_plots:
    print(f"Found {len(spall_plots)} spall plots — showing the first one:")
    ipy_display(Image(spall_plots[0], width=800))
else:
    print("No spall plots found — check that plot_individual: true in the config.")